# Skripsi: Integrasi Confident Learning dan CORN pada IndoBERT
## Pilot Study — Deteksi Label Noise pada Prediksi Rating Ulasan Google Play Store

**Ryan Besto Saragih — NIM 23051204205**
Pembimbing: Anita Qoiriah, S.Kom., M.Kom.

Notebook ini menjalankan pipeline lengkap pilot study sesuai Bab 1 (Subbab 1.2) dan
Bab 3 proposal: scraping → praproses → ablasi proxy classifier (P1-P4) →
deteksi noise → validasi manusia → training 6 skenario model → uji signifikansi
statistik (4 hipotesis pre-registered).

**Struktur data**: satu skema tunggal (tidak ada versioning) — `all_reviews_master.csv`
dikunci setelah scraping selesai dan dipakai sebagai sumber kebenaran tunggal untuk
seluruh eksperimen berikutnya, sesuai Batasan Masalah Bab 1.6.

**Cara pakai**: jalankan cell berurutan dari atas ke bawah. Cell scraping (Cell 3)
otomatis dilewati kalau data sudah ada — jangan hapus `all_reviews_master.csv`
kecuali memang berniat scraping ulang dari nol.

In [ ]:
# ==========================================================
# CELL 2: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/skripsi-corn-label-noise"
REPO_DIR = "/content/skripsi-corn-label-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

## Instalasi Dependencies

Termasuk `py3langid` untuk filter bahasa saat scraping. Pastikan `requirements.txt`
di repo sudah memuat: `pandas`, `numpy`, `scikit-learn`, `torch`, `transformers`,
`cleanlab`, `coral-pytorch`, `statsmodels`, `scipy`, `google-play-scraper`,
`py3langid`.

In [ ]:
# ==========================================================
# CELL 4: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

## Import Umum
Dipakai berulang di banyak cell berikutnya — diletakkan sekali di sini supaya
tidak diulang-ulang di tiap cell.

In [ ]:
# ==========================================================
# CELL 6: IMPORT UMUM
# ==========================================================
import os
import pandas as pd
import numpy as np

from src import config
print(f"📌 Proxy default aktif: [{config.PROXY_ID}] {config.PROXY_NAME}")
print(f"📁 Drive root: {config.DRIVE_ROOT}")

## Scraping Google Play Store

Scraping dengan filter bahasa Indonesia inline (`py3langid`), target per rating
sudah dinaikkan untuk mengantisipasi susut data sepanjang pipeline (praproses →
split → K-Fold → pruning), dan dilengkapi laporan kualitas data (distribusi
tanggal per app/rating untuk cek confound temporal, distribusi panjang teks,
deteksi teks duplikat) yang otomatis tersimpan di `results/` untuk dilampirkan
di Bab III dan Bab IV.

⚠️ **JALANKAN SEKALI SAJA SEUMUR PROYEK.** Setelah selesai, `all_reviews_master.csv`
dikunci dan dipakai sebagai sumber kebenaran tunggal. Cell ini **otomatis dilewati**
kalau file tersebut sudah ada — hapus manual hanya kalau memang sengaja ingin
scraping ulang dari nol (dan sadar semua eksperimen berikutnya perlu diulang).

Estimasi waktu: bisa 1-3 jam tergantung koneksi dan seberapa jauh Google Play
Store perlu ditelusuri untuk rating 2/3 yang jarang muncul.

In [ ]:
# ==========================================================
# CELL 8: SCRAPING GOOGLE PLAY STORE (AUTO-SKIP KALAU SUDAH ADA)
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
    df_check = pd.read_csv(config.RAW_DATA_FILE)
    print(f"   Baris tersedia: {len(df_check)}")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

## Praproses Teks

Case folding, penghapusan URL/username, translasi emoji ke sentimen (kamus
diperluas), normalisasi elongasi huruf, normalisasi slang (Kamus Alay + kamus
domain, dengan perbaikan pencocokan token bertanda-baca), resolusi konflik
teks-identik-rating-berbeda (voting mayoritas), dan penghapusan duplikat murni.

Preprocessing dijaga minimal (tanpa stemming/stopword removal) sesuai rekomendasi
untuk model berbasis BERT (Wilie dkk. 2020; Koto dkk. 2020).

⚠️ **JALANKAN SEKALI SAJA.** Split train/test yang dihasilkan (`split_train_raw.csv`
/ `split_test.csv`) dikunci dan dipakai di seluruh eksperimen berikutnya, termasuk
pilot study proxy P1-P4. Cell ini otomatis dilewati kalau split sudah ada.

In [ ]:
# ==========================================================
# CELL 10: PREPROCESSING + TRAIN-TEST SPLIT (SUMBER KEBENARAN TUNGGAL)
# ==========================================================
from sklearn.model_selection import train_test_split
from src.preprocess import run_preprocessing
from src import config

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.TEST_FILE):
    print("✅ Split train/test sudah ada -- preprocessing dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} baris | Test: {len(df_test)} baris")
else:
    print("=" * 60)
    print(" PREPROCESSING ")
    print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) ")
    print("=" * 60)
    df_train, df_test = train_test_split(
        df_clean, test_size=0.2, random_state=42, stratify=df_clean["rating"]
    )
    df_train.to_csv(config.TRAIN_RAW_FILE, index=False)
    df_test.to_csv(config.TEST_FILE, index=False)

    print(f"✅ Train: {len(df_train)} baris -> {config.TRAIN_RAW_FILE}")
    print(f"✅ Test : {len(df_test)} baris -> {config.TEST_FILE}")

In [ ]:
# ==========================================================
# CELL 10B: SAMPLING SUBSET UJI EMAS (GOLD TEST SUBSET) -- Subbab 3.9.4
# ==========================================================
from src import config, gold_test

if os.path.exists(config.GOLD_TEST_SAMPLE_FILE):
    print(f"✅ {config.GOLD_TEST_SAMPLE_FILE} sudah ada -- sampling dilewati.")
else:
    gold_test.sample_gold_test_subset(seed=42)
    gold_test.export_second_annotator_gold_subset()

print("\n⚠️  Anotasi gold test bisa dikerjakan PARALEL sambil menunggu training")
print("   6 skenario (Cell 20) berjalan -- evaluasinya baru dijalankan di Cell 23")
print("   setelah training tuntas. Isi 'human_gold_rating' (1-5) langsung di file di atas.")

## Pilot Study — Ablasi Proxy Classifier (P1-P4)

Menjalankan Confident Learning untuk 4 dari 5 tahap ablasi proxy classifier
(Tabel 3.2, Bab 3): P1 (CLS embedding beku + LR), P2 (mean-pooling beku + LR),
P3 (IndoBERT fine-tuned, Cross-Entropy), P4 (IndoBERT fine-tuned, CORN — proxy
final). P5 (fusi sentimen eksternal) dijalankan terpisah di Cell berikutnya
karena butuh model sentimen tambahan dan bukan bagian pilot study Bab 1.

Setiap proxy sekarang: (a) memakai skema K-Fold yang identik (shuffle=True,
seed=42) untuk P1/P2 maupun P3/P4, (b) dikalibrasi lewat temperature scaling
untuk P3/P4, (c) metode filter cleanlab dipilih otomatis lewat kriteria
eksplisit (paling dekat dengan estimasi teoretis `num_label_issues()`), bukan
hardcode.

Hasilnya terkumpul di `results/proxy_ablation_table.csv` — inilah **Tabel 1.1**
pilot study di Bab 1 proposal.

⏱️ P3 dan P4 butuh fine-tuning K-Fold (5 fold × 3 epoch masing-masing) — cell
ini bisa memakan waktu cukup lama.

In [ ]:
# ==========================================================
# CELL 12: PILOT STUDY -- ABLASI PROXY P1-P4
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

PILOT_PROXY_IDS = [0, 1, 2, 3]  # P5 (fusion, id 4) dijalankan terpisah di Cell 14

for pid in PILOT_PROXY_IDS:
    print(f"\n{'='*70}\n PILOT STUDY -- PROXY_ID = {pid} \n{'='*70}")
    config.set_proxy(pid)
    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Pilot study (P1-P4) selesai.")
pilot_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(pilot_table)

## Ablasi Tambahan — P5 (Fusi Sentimen Eksternal)

P5 **bukan** bagian dari pilot study Bab 1, tapi bagian dari studi ablasi lima
tahap penuh (Tabel 3.2) yang dilaporkan di Bab IV. Menambah satu baris ke
`proxy_ablation_table.csv` (proxy_id=4). Model fusion sekarang memproyeksikan
skor sentimen (3-dim) ke ruang 32-dim sebelum digabung dengan representasi
BERT (768-dim), supaya sinyalnya tidak tenggelam secara numerik (lihat
`models.py`).

⚠️ Cell ini **wajib diakhiri** dengan `config.set_proxy(3)` — supaya proxy aktif
kembali ke final (P4) sebelum cell manapun setelah ini dijalankan.

In [ ]:
# ==========================================================
# CELL 14: ABLASI TAMBAHAN -- P5 (FUSI SENTIMEN)
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_proxy(4)
try:
    df_noise_p5, proxy_metrics_p5 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 4 (fusion) gagal:")
    traceback.print_exc()
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke final: [{config.PROXY_ID}] {config.PROXY_NAME}")

## Kunci Proxy Final (P4) + Ekspor Sample Validasi Manusia

Menjalankan ulang Confident Learning dengan proxy final terkunci (P4/CORN,
sesuai skema integrasi ketat Bab 1.1) — akan memuat dari cache kalau sudah
pernah dihitung di Cell 12, jadi cepat. Ini menghasilkan sample 50 baris
(stratified by rating_diff) untuk validasi manusia.

⚠️ **BERHENTI setelah cell ini** — isi dulu `human_verdict` secara manual
sebelum lanjut ke Cell 17.

In [ ]:
# ==========================================================
# CELL 16: KUNCI PROXY FINAL (P4) -- WAJIB SEBELUM VALIDASI MANUSIA
# ==========================================================
from src import config

config.set_proxy(3)
print(f"🔒 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

print("\n⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 17 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan Cell 17 untuk cek kelengkapan + hitung agreement rate.")

## Hitung Agreement Rate Validasi Manusia

Menjalankan **setelah** kamu selesai mengisi `human_verdict` secara manual di
file yang disebut Cell 16. Melaporkan agreement rate keseluruhan dan per-bin
`rating_diff` — breakdown per-bin ini penting sebagai bukti kualitatif
pendukung/penolak asumsi ambang severity-aware pruning (Subbab 3.7).

In [ ]:
from src import config
import pandas as pd

df_check = pd.read_csv(config.HUMAN_VALIDATION_FILE, sep=None, engine="python")
print(df_check.shape)
print(df_check["human_verdict"].value_counts(dropna=False))

In [ ]:
# ==========================================================
# CELL 18: HITUNG AGREEMENT VALIDASI MANUSIA
# ==========================================================
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 19 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

In [ ]:
# ==========================================================
# CELL 18B: ANOTATOR KEDUA + COHEN'S KAPPA -- VALIDASI NOISE (Subbab 3.9.3)
# ==========================================================
from src import human_validation as hv

hv.export_second_annotator_subset()
print("\n⚠️  Minta penilai kedua mengisi 'human_verdict' pada file di atas, BUTA seperti anotator pertama.")
print("Setelah selesai, jalankan:")
print(">>> from src import human_validation as hv")
print(">>> hv.compute_interannotator_kappa()")
print("\nKalau hanya ada satu penilai (fallback single-annotator), jalankan sebagai gantinya:")
print(">>> hv.export_retest_subset()")
print(">>> # isi manual, lalu:")
print(">>> hv.compute_test_retest_reliability()")

## Training 6 Skenario × 3 Seed

Melatih 6 skenario (kombinasi 3 varian data × 2 fungsi loss): M1-M3 (raw/
hard-prune/severity-prune × Cross-Entropy), M4-M6 (raw/hard-prune/severity-prune
× CORN). Setiap skenario dilatih pada 3 seed (42, 123, 2024) dengan
mixed-precision training dan early stopping (berhenti kalau Val MAE tidak
membaik 3 epoch berturut-turut).

Auto-resume: kalau sesi Colab terputus, jalankan ulang cell ini — training
akan melanjutkan dari skenario/seed terakhir yang belum selesai, bukan mulai
dari nol.

⏱️ Ini cell paling lama di seluruh notebook — bisa beberapa jam tergantung GPU
yang dialokasikan Colab.

In [ ]:
# ==========================================================
# CELL 20: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

print(f"🔧 Training M1-M6 akan pakai backbone: {config.PRETRAINED_MODEL_NAME} | "
      f"proxy final: {config.PROXY_NAME}")

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(" 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆")
print("=" * 100)
display(df_final)

## Uji Signifikansi — Wilcoxon + Holm-Bonferroni (4 Hipotesis)

Menguji 4 hipotesis pre-registered (Subbab 3.9.2):
- **H1**: CORN vs CE pada data mentah (M4 vs M1)
- **H2**: severity-aware vs baseline pada CORN (M6 vs M4)
- **H3**: hard-prune vs baseline pada CORN (M5 vs M4)
- **H4**: severity-aware vs hard-prune langsung (M6 vs M5) — menjawab langsung
  pertanyaan rumusan masalah Bab 1.3 soal mana strategi cleaning yang lebih baik,
  bukan disimpulkan tidak langsung dari H2 dan H3

Prediksi dikumpulkan dari **ketiga seed** (bukan cuma seed 42), diagregasi
per sampel, baru diuji — sesuai janji Subbab 3.9.2. Effect size + CI 95%
(bootstrap) dilaporkan sebagai pelengkap p-value, disimpan otomatis ke CSV.

In [ ]:
# ==========================================================
# CELL 22: UJI SIGNIFIKANSI -- WILCOXON + HOLM-BONFERRONI (4 HIPOTESIS)
# ==========================================================
from src.significance import (
    collect_all_predictions,
    aggregate_errors_across_seeds,
    run_significance_test,
    run_all_effect_sizes,
)
from src import config

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

print("\n📊 Mengagregasi absolute error across seed...")
aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Uji Wilcoxon Signed-Rank (4 hipotesis pre-registered) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)

effect_sizes_file = os.path.join(config.RESULTS_DIR, "effect_sizes.csv")
effect_sizes.to_csv(effect_sizes_file, index=False)
print(f"\n💾 Effect sizes -> {effect_sizes_file}")

In [ ]:
# ==========================================================
# CELL 23: EVALUASI 6 SKENARIO PADA SUBSET UJI EMAS -- Subbab 3.9.4
# ==========================================================
from src import config, gold_test

result = gold_test.evaluate_on_gold_subset(scenarios)
if result is not None:
    print("\n✅ Evaluasi subset uji emas selesai -- lampirkan ke Bab IV berdampingan "
          "dengan hasil data uji penuh (Cell 20 & 22).")
    gold_test.compute_gold_interannotator_kappa()
else:
    print(f"\n⛔ Lengkapi dulu {config.GOLD_TEST_SAMPLE_FILE} sebelum menjalankan cell ini.")

## Ringkasan Akhir — Siap Disalin ke Bab 1 (Pilot Study) dan Bab 4

Semua dibaca dari file di Drive (bukan variabel session), jadi cell ini bisa
dijalankan kapan saja, tidak peduli cell mana yang sudah/belum jalan di sesi
Colab saat ini.

In [ ]:
# ==========================================================
# CELL 24: RINGKASAN AKHIR
# ==========================================================
import pandas as pd
from src import config

print("=" * 70)
print(" RINGKASAN LENGKAP ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (P1-P5) -- untuk Bab 1 (pilot study P1-P4) & Bab 4 (lengkap P1-P5):")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    display(pd.read_csv(config.PROXY_QUALITY_LOG_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 12 dulu.")

print("\n[2] LAPORAN KUALITAS DATA SCRAPING -- untuk Bab 3:")
for fname in ["scraping_summary.csv", "date_distribution_per_app_rating.csv",
              "text_length_summary.csv", "duplicate_review_texts.csv"]:
    fpath = os.path.join(config.RESULTS_DIR, fname)
    status = "✅" if os.path.exists(fpath) else "⚠️ belum ada"
    print(f"   {status} {fpath}")

print("\n[3] VALIDASI MANUSIA -- untuk Bab 4:")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    df_human = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
    counts = df_human["human_verdict"].value_counts()
    total = len(df_human)
    print(f"   Total: {total} | Noise: {counts.get('noise',0)} ({counts.get('noise',0)/total*100:.1f}%) "
          f"| Not noise: {counts.get('not_noise',0)} ({counts.get('not_noise',0)/total*100:.1f}%) "
          f"| Ambiguous: {counts.get('ambiguous',0)} ({counts.get('ambiguous',0)/total*100:.1f}%)")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 18 dulu.")

print("\n[4] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:")
if os.path.exists(config.FINAL_RESULTS_TABLE_FILE):
    display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 20 dulu.")

print("\n[5] UJI SIGNIFIKANSI (4 hipotesis pre-registered) -- untuk Bab 4:")
if os.path.exists(config.SIGNIFICANCE_TEST_FILE):
    display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n[6] EFFECT SIZE + CI 95% -- untuk Bab 4:")
effect_size_file = os.path.join(config.RESULTS_DIR, "effect_sizes.csv")
if os.path.exists(effect_size_file):
    display(pd.read_csv(effect_size_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)